In [1]:
import html

import ipywidgets as widgets


def create_test_widget(js_code, title):
    html_content = f"""
    <html>
    <head>
        <style>body {{ font-family: sans-serif; padding: 20px; }}</style>
    </head>
    <body>
        <h3>{title}</h3>
        <div id="result">Running...</div>
        <script>
            try {{
                {js_code}
            }} catch (e) {{
                document.getElementById('result').innerText = 'Error: ' + e.message;
            }}
        </script>
    </body>
    </html>
    """
    escaped_html = html.escape(html_content)
    iframe_html = (
        f'<iframe srcdoc="{escaped_html}" '
        f'width="100%" height="150" frameborder="1" '
        f'sandbox="allow-scripts allow-same-origin allow-popups" '
        f"></iframe>"
    )
    return widgets.HTML(value=iframe_html)

## Test 1: JupyterLab CSS Variables
Try to read JupyterLab's main layout CSS variables

In [ ]:
js = """
const parentDoc = window.parent?.document;
const rootStyle = parentDoc ? getComputedStyle(parentDoc.documentElement) : null;
const results = [];

// JupyterLab main layout colors
const jpVars = [
    '--jp-layout-color0',
    '--jp-layout-color1', 
    '--jp-layout-color2',
    '--jp-cell-editor-background',
    '--jp-notebook-background',
    '--jp-editor-background',
    '--jp-content-font-color1'
];

jpVars.forEach(v => {
    const val = rootStyle?.getPropertyValue(v)?.trim();
    const colorBox = val ? '<span style="display:inline-block;width:20px;height:12px;border:1px solid #888;background:' + val + ';vertical-align:middle;margin-right:8px;"></span>' : '';
    results.push(colorBox + v + ': ' + (val || 'Not found'));
});

document.getElementById('result').innerHTML = results.join('<br>');
"""
create_test_widget(js, "Test 1: JupyterLab CSS Variables (Parent)")

HTML(value='<iframe srcdoc="\n    &lt;html&gt;\n    &lt;head&gt;\n        &lt;style&gt;body { font-family: san…

## Test 2: JupyterLab Body Data Attributes
Check for JupyterLab-specific data attributes on the body

In [3]:
js = """
const parentDoc = window.parent?.document;
const results = [];

if (parentDoc) {
    // All data attributes on body
    const bodyDataset = parentDoc.body.dataset;
    results.push('Body data attributes:');
    for (const [key, value] of Object.entries(bodyDataset)) {
        results.push('  data-' + key + ': ' + value);
    }
    
    // Body classes
    results.push('');
    results.push('Body classes: ' + parentDoc.body.className);
    
    // Check specific JupyterLab attributes
    results.push('');
    results.push('data-jp-theme-light: ' + (bodyDataset.jpThemeLight || 'Not found'));
    results.push('data-jp-theme-name: ' + (bodyDataset.jpThemeName || 'Not found'));
} else {
    results.push('Cannot access parent document');
}

document.getElementById('result').innerHTML = results.join('<br>');
"""
create_test_widget(js, "Test 2: JupyterLab Body Data Attributes")

HTML(value='<iframe srcdoc="\n    &lt;html&gt;\n    &lt;head&gt;\n        &lt;style&gt;body { font-family: san…

## Test 3: Body Background Color
Get the computed background color of the parent body

In [ ]:
js = """
const parentDoc = window.parent?.document;
const results = [];

if (parentDoc) {
    const bodyStyle = getComputedStyle(parentDoc.body);
    const htmlStyle = getComputedStyle(parentDoc.documentElement);
    
    const colorBox = (c) => '<span style="display:inline-block;width:20px;height:12px;border:1px solid #888;background:' + c + ';vertical-align:middle;margin-right:8px;"></span>';
    results.push(colorBox(bodyStyle.backgroundColor) + 'body.backgroundColor: ' + bodyStyle.backgroundColor);
    results.push(colorBox(htmlStyle.backgroundColor) + 'html.backgroundColor: ' + htmlStyle.backgroundColor);
    results.push(colorBox(bodyStyle.color) + 'body.color: ' + bodyStyle.color);
    
    // Try to find the main content area
    const mainPanel = parentDoc.querySelector('#jp-main-dock-panel');
    if (mainPanel) {
        const panelStyle = getComputedStyle(mainPanel);
        results.push(colorBox(panelStyle.backgroundColor) + '#jp-main-dock-panel bg: ' + panelStyle.backgroundColor);
    }
    
    const notebookPanel = parentDoc.querySelector('.jp-Notebook');
    if (notebookPanel) {
        const nbStyle = getComputedStyle(notebookPanel);
        results.push(colorBox(nbStyle.backgroundColor) + '.jp-Notebook bg: ' + nbStyle.backgroundColor);
    }
    
    const cellArea = parentDoc.querySelector('.jp-Cell');
    if (cellArea) {
        const cellStyle = getComputedStyle(cellArea);
        results.push(colorBox(cellStyle.backgroundColor) + '.jp-Cell bg: ' + cellStyle.backgroundColor);
    }
} else {
    results.push('Cannot access parent document');
}

document.getElementById('result').innerHTML = results.join('<br>');
"""
create_test_widget(js, "Test 3: Body/Element Background Colors")

HTML(value='<iframe srcdoc="\n    &lt;html&gt;\n    &lt;head&gt;\n        &lt;style&gt;body { font-family: san…

## Test 4: color-scheme CSS Property
Check if JupyterLab sets the color-scheme property

In [5]:
js = """
const parentDoc = window.parent?.document;
const results = [];

if (parentDoc) {
    const htmlStyle = getComputedStyle(parentDoc.documentElement);
    const bodyStyle = getComputedStyle(parentDoc.body);
    
    results.push('html color-scheme: ' + htmlStyle.getPropertyValue('color-scheme'));
    results.push('body color-scheme: ' + bodyStyle.getPropertyValue('color-scheme'));
    
    // Also check for prefers-color-scheme
    const prefersDark = window.matchMedia('(prefers-color-scheme: dark)').matches;
    const prefersLight = window.matchMedia('(prefers-color-scheme: light)').matches;
    results.push('');
    results.push('matchMedia prefers-dark: ' + prefersDark);
    results.push('matchMedia prefers-light: ' + prefersLight);
} else {
    results.push('Cannot access parent document');
}

document.getElementById('result').innerHTML = results.join('<br>');
"""
create_test_widget(js, "Test 4: color-scheme CSS Property")

HTML(value='<iframe srcdoc="\n    &lt;html&gt;\n    &lt;head&gt;\n        &lt;style&gt;body { font-family: san…

## Test 5: JupyterLab Shell Classes
Check for theme-related classes on the JupyterLab shell

In [6]:
js = """
const parentDoc = window.parent?.document;
const results = [];

if (parentDoc) {
    // Check various JupyterLab elements for classes
    const selectors = [
        'body',
        'html',
        '#main',
        '.jp-LabShell',
        '.jp-Application'
    ];
    
    selectors.forEach(sel => {
        const el = parentDoc.querySelector(sel);
        if (el) {
            results.push(sel + ' classes: ' + (el.className || '(none)'));
        } else {
            results.push(sel + ': not found');
        }
    });
} else {
    results.push('Cannot access parent document');
}

document.getElementById('result').innerHTML = results.join('<br>');
"""
create_test_widget(js, "Test 5: JupyterLab Shell Classes")

HTML(value='<iframe srcdoc="\n    &lt;html&gt;\n    &lt;head&gt;\n        &lt;style&gt;body { font-family: san…

## Test 6: Comprehensive Detection (Current Implementation)
Shows what our current detection would return

In [ ]:
js = """
const parseColorString = (value) => {
    if (!value) return null;
    const scratch = document.createElement('div');
    scratch.style.color = value;
    scratch.style.backgroundColor = value;
    scratch.style.display = 'none';
    document.body.appendChild(scratch);
    const resolved = getComputedStyle(scratch).color || '';
    scratch.remove();
    const nums = resolved.match(/[\\d\\.]+/g);
    if (nums && nums.length >= 3) {
        const [r, g, b] = nums.slice(0, 3).map(Number);
        if (nums.length >= 4) {
            const alpha = Number(nums[3]);
            if (alpha < 0.1) return null;
        }
        const luminance = 0.299 * r + 0.587 * g + 0.114 * b;
        return { r, g, b, luminance, resolved };
    }
    return null;
};

const results = [];
const parentDoc = window.parent?.document;

// 1. Try JupyterLab CSS variables first
if (parentDoc) {
    const rootStyle = getComputedStyle(parentDoc.documentElement);
    const jpLayoutColor0 = rootStyle.getPropertyValue('--jp-layout-color0')?.trim();
    const jpLayoutColor1 = rootStyle.getPropertyValue('--jp-layout-color1')?.trim();
    
    const colorBox = (c) => c ? '<span style="display:inline-block;width:20px;height:12px;border:1px solid #888;background:' + c + ';vertical-align:middle;margin-right:8px;"></span>' : '';
    results.push(colorBox(jpLayoutColor0) + '--jp-layout-color0: ' + (jpLayoutColor0 || 'not found'));
    results.push(colorBox(jpLayoutColor1) + '--jp-layout-color1: ' + (jpLayoutColor1 || 'not found'));
    
    if (jpLayoutColor0) {
        const parsed = parseColorString(jpLayoutColor0);
        if (parsed) {
            results.push('  Parsed: rgb(' + parsed.r + ',' + parsed.g + ',' + parsed.b + ')');
            results.push('  Luminance: ' + parsed.luminance.toFixed(1));
            results.push('  Theme: ' + (parsed.luminance > 150 ? 'LIGHT' : 'DARK'));
        }
    }
    
    // 2. JupyterLab data attribute
    results.push('');
    const jpThemeLight = parentDoc.body.dataset.jpThemeLight;
    results.push('data-jp-theme-light: ' + (jpThemeLight || 'not found'));
    if (jpThemeLight) {
        results.push('  Theme from attr: ' + (jpThemeLight === 'true' ? 'LIGHT' : 'DARK'));
    }
}

document.getElementById('result').innerHTML = results.join('<br>');
"""
create_test_widget(js, "Test 6: Comprehensive Detection")

HTML(value='<iframe srcdoc="\n    &lt;html&gt;\n    &lt;head&gt;\n        &lt;style&gt;body { font-family: san…

## Test 7: Find Best Background Color Source
Tries multiple sources to find the actual visible background

In [ ]:
js = """
const results = [];
const parentDoc = window.parent?.document;

const getCSSVar = (name) => {
    if (!parentDoc) return null;
    return getComputedStyle(parentDoc.documentElement).getPropertyValue(name)?.trim() || null;
};

const getElementBg = (selector) => {
    if (!parentDoc) return null;
    const el = parentDoc.querySelector(selector);
    if (!el) return null;
    const bg = getComputedStyle(el).backgroundColor;
    // Skip transparent/rgba(0,0,0,0)
    if (!bg || bg === 'transparent' || bg === 'rgba(0, 0, 0, 0)') return null;
    return bg;
};

// Priority order for JupyterLab background color
const sources = [
    { name: '--jp-layout-color0', value: getCSSVar('--jp-layout-color0') },
    { name: '--jp-layout-color1', value: getCSSVar('--jp-layout-color1') },
    { name: '--jp-notebook-background', value: getCSSVar('--jp-notebook-background') },
    { name: '--jp-cell-editor-background', value: getCSSVar('--jp-cell-editor-background') },
    { name: '.jp-Notebook bg', value: getElementBg('.jp-Notebook') },
    { name: '.jp-Cell bg', value: getElementBg('.jp-Cell') },
    { name: '#jp-main-dock-panel bg', value: getElementBg('#jp-main-dock-panel') },
    { name: 'body.backgroundColor', value: getElementBg('body') },
];

results.push('Background color sources (in priority order):');
results.push('');

const colorBox = (c) => c ? '<span style="display:inline-block;width:20px;height:12px;border:1px solid #888;background:' + c + ';vertical-align:middle;margin-right:8px;"></span>' : '';

let firstValid = null;
sources.forEach(s => {
    const status = s.value ? '✓' : '✗';
    results.push(colorBox(s.value) + status + ' ' + s.name + ': ' + (s.value || 'not found/transparent'));
    if (s.value && !firstValid) firstValid = s;
});

results.push('');
results.push(colorBox(firstValid?.value) + '<b>BEST SOURCE: ' + (firstValid ? firstValid.name + ' = ' + firstValid.value : 'NONE') + '</b>');

document.getElementById('result').innerHTML = results.join('<br>');

"""create_test_widget(js, "Test 7: Find Best Background Color Source")

HTML(value='<iframe srcdoc="\n    &lt;html&gt;\n    &lt;head&gt;\n        &lt;style&gt;body { font-family: san…